# Feature Comparison: Baseline vs Extended

Does **hair removal** actually change the extracted feature **values**?

We compare two feature sets:
- **baseline** — no hair removal, no downscaling
- **extended** — same images, but with inpainting-based hair removal

Because both pipelines run on the **same images**, we can compare distributions directly, and also compute **per-image differences** (the most sensitive test).

## 0. Imports & load

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FIG_DIR = './results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

PIPELINES = {
    'baseline': 'src/featureDf_baseline.csv',
    'extended': 'src/featureDf_extended.csv',
}

dfs = {name: pd.read_csv(path) for name, path in PIPELINES.items()}
for name, df in dfs.items():
    print(f'{name:10s}: {df.shape[0]} rows, {df.shape[1]} cols')

In [ ]:
# Auto-detect feature columns (everything except metadata / status columns)
exclude = ['img_id', 'patient_id', 'diagnostic', 'label',
           'processing_status', 'error_message']
feature_cols = [c for c in dfs['baseline'].columns if c not in exclude]
print(f'{len(feature_cols)} feature columns:')
print(feature_cols)

## 1. Overlaid density plots (baseline vs extended)

Each subplot overlays the distribution of one feature for the two pipelines.

**How to read:**
- Curves **overlap** → hair removal did **not** change this feature.
- Curves **shift / change shape** → hair removal altered the feature values.

Expectation: **shape features** (`symmetry`, `border`, `diameter`) should overlap, because hair removal changes pixels but not the lesion **mask** they are computed from. **Colour features** may shift, because inpainting alters pixel colours.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(feature_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols):
    for name, df in dfs.items():
        df[col].dropna().plot.kde(ax=ax, label=name, lw=2)
    ax.set_title(col)
    ax.legend(fontsize=8)
    ax.set_ylabel('')

# Hide any unused axes
for ax in axes[len(feature_cols):]:
    ax.axis('off')

plt.suptitle('Feature distributions: baseline vs extended', y=1.001, fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/feature_density_baseline_vs_extended.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Paired per-image differences (extended − baseline)

Because the same images go through both pipelines, we can subtract feature values **image by image**. This removes between-image noise and is far more sensitive than comparing two separate distributions.

**How to read:**
- A sharp spike centred on **0** → hair removal barely changed this feature.
- A spread-out or **shifted** curve → hair removal systematically changed it.

In [ ]:
# Align baseline and extended on img_id so the subtraction is paired
paired = dfs['baseline'].merge(dfs['extended'], on='img_id', suffixes=('_base', '_ext'))
print(f'Paired images: {len(paired)}')

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols):
    delta = (paired[f'{col}_ext'] - paired[f'{col}_base']).dropna()
    if delta.std() > 1e-9:
        delta.plot.kde(ax=ax, lw=2, color='tab:orange')
    else:
        ax.hist(delta, bins=1, color='tab:orange')  # all-zero diffs
    ax.axvline(0, color='k', ls='--', alpha=0.6)
    ax.set_title(f'{col}  (Δ ext−base)')
    ax.set_ylabel('')

for ax in axes[len(feature_cols):]:
    ax.axis('off')

plt.suptitle('Per-image feature change after hair removal (extended − baseline)', y=1.001, fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/feature_delta_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Summary table: how much did each feature change?

Quantifies the per-image change. Features are sorted by **mean absolute change** — the ones at the top were most affected by hair removal.

In [ ]:
rows = []
for col in feature_cols:
    delta = (paired[f'{col}_ext'] - paired[f'{col}_base']).dropna()
    rows.append({
        'feature':         col,
        'mean_baseline':   paired[f'{col}_base'].mean(),
        'mean_extended':   paired[f'{col}_ext'].mean(),
        'mean_abs_change': delta.abs().mean(),
        'max_abs_change':  delta.abs().max(),
        'pct_changed':     (delta.abs() > 1e-6).mean() * 100,
    })

summary = pd.DataFrame(rows).sort_values('mean_abs_change', ascending=False).reset_index(drop=True)
summary.round(4)

In [ ]:
# Save summary for the report
os.makedirs('./results/reports', exist_ok=True)
summary.round(5).to_csv('./results/reports/feature_change_baseline_vs_extended.csv', index=False)
print('Saved to ./results/reports/feature_change_baseline_vs_extended.csv')

## 4. Statistical test: is the change significant?

We have **paired** observations (the same image through both pipelines), so we apply the **Wilcoxon signed-rank test** to the per-image differences of each feature.

- **H0**: the median difference (extended − baseline) is 0 → hair removal did not change this feature.
- **p < α** → reject H0 → the change is statistically significant.
- We use Wilcoxon (non-parametric) rather than a paired t-test because the differences are far from normal (sharply peaked at 0 with heavy tails).

**Multiple comparisons:** we test 15 features at once, so we also report a **Bonferroni-corrected** threshold (α / 15) to control false positives.

**⚠️ Large-sample caveat:** with N ≈ 1,945 paired images, statistical power is very high — even a tiny, practically irrelevant shift can come out "significant". Always read the p-value **together with** the effect size (`mean_abs_change`).

In [ ]:
from scipy.stats import wilcoxon

ALPHA = 0.05
n_tests = len(feature_cols)
alpha_bonf = ALPHA / n_tests   # Bonferroni-corrected threshold

rows = []
for col in feature_cols:
    delta = (paired[f'{col}_ext'] - paired[f'{col}_base']).dropna()
    n_changed = int((delta.abs() > 1e-12).sum())

    if n_changed == 0:
        # All per-image differences are exactly 0 -> no change, test not applicable
        rows.append({'feature': col, 'n_changed': 0, 'mean_abs_change': 0.0,
                     'wilcoxon_p': np.nan, 'sig_alpha': False, 'sig_bonferroni': False})
        continue

    stat, p = wilcoxon(delta)   # H0: median difference = 0
    rows.append({
        'feature':         col,
        'n_changed':       n_changed,
        'mean_abs_change': delta.abs().mean(),
        'wilcoxon_p':      p,
        'sig_alpha':       bool(p < ALPHA),
        'sig_bonferroni':  bool(p < alpha_bonf),
    })

test_df = pd.DataFrame(rows).sort_values('mean_abs_change', ascending=False).reset_index(drop=True)
print(f'N = {len(paired)} paired images | alpha = {ALPHA} | Bonferroni alpha = {alpha_bonf:.4f} ({n_tests} tests)')
test_df.round(6)

In [ ]:
# Save the statistical test results for the report
test_df.round(6).to_csv('./results/reports/feature_change_significance.csv', index=False)
print('Saved to ./results/reports/feature_change_significance.csv')

n_sig = int(test_df['sig_bonferroni'].sum())
n_unchanged = int((test_df['n_changed'] == 0).sum())
print(f'\n{n_sig}/{n_tests} features significantly changed (Bonferroni).')
print(f'{n_unchanged}/{n_tests} features identical in every image (Δ = 0 everywhere).')

### How to read the test table

| Column | Meaning |
|---|---|
| `n_changed` | how many of the ~1,945 images had a non-zero difference |
| `mean_abs_change` | **effect size** — average magnitude of the change |
| `wilcoxon_p` | p-value for H0 "no change" |
| `sig_alpha` | significant at α = 0.05 |
| `sig_bonferroni` | significant after correcting for 15 tests |

**Reading guide:**
- **Shape features** (`symmetry`, `border`, `diameter`): `n_changed = 0`, no p-value — they are identical in every image (hair removal leaves the mask untouched). Sanity check passed.
- **Colour features**: likely `sig_bonferroni = True` (significant), because with N ≈ 1,945 even small shifts reach significance. **This is why the effect size matters** — a feature can be "significantly" changed yet by a practically tiny amount.

**What to write in the report:**
> *"A paired Wilcoxon signed-rank test confirmed that hair removal significantly altered the colour features (p < 0.05, Bonferroni-corrected), while the shape features were unchanged in every image. However, the effect sizes were small (mean absolute change ≈ X), indicating that although the change is statistically detectable on ~1,900 paired images, its practical magnitude is limited — consistent with the modest decrease in classification AUC."*

## How to interpret / what to write in the report

- **Shape features** (`symmetry`, `border`, `diameter`) should show `mean_abs_change ≈ 0` and `pct_changed ≈ 0%` — they are computed from the lesion mask, which hair removal does not touch. This is a useful sanity check.
- **Colour features** with the largest `mean_abs_change` are the ones inpainting affected most. These are the likely drivers of the AUC difference between baseline and extended.
- If a colour feature barely changes, hair removal had little effect on it; if it changes a lot, that feature carries the impact of the hair-removal step.

**Example sentence:** *"Per-image comparison confirmed that hair removal left the shape features (symmetry, border, diameter) unchanged (mean absolute change = 0), while colour features — particularly `<top feature>` — shifted substantially, explaining why the extended pipeline produced different model performance."*